# IIT-4. Le problème de frontière — qui décide où sont les bords ?

**Série** : Integrated Information Theory avec PyPhi (suite de [IIT-3 — coarse-graining et échelle du $\Phi$](IIT-3-CoarseGrainingMacroPhi.ipynb))
**Outil** : PyPhi 1.2.0 — le vrai moteur IIT (identification, MIP, $\Phi$), pas une ré-implémentation.

Toute la machinerie de recollement du dépôt suppose un recouvrement **donné** : des contextes
$U_i$, leurs chevauchements, et la question de savoir si les sections locales se recollent. Ce
notebook pose la question d'avant :

> **Avant de demander comment des sections recollent — QUI DÉCIDE OÙ SE TROUVENT LES BORDS DES SECTIONS ?**

C'est le *boundary problem* (voir la proposition de Gómez Emilsson, citée en référence **comme
proposition** — sa réponse propre, une topologie de champs électromagnétiques, n'est ni un fait
acquis ni la nôtre ; nous ne l'importons pas). Le problème, lui, est exactement ce qui manque en
amont : un découpage n'est jamais donné par la physique des données, c'est un **choix**.

Or PyPhi partitionne en permanence — et la partition **est** le choix de frontière. Chaque appel à
`Subsystem(network, state, nodes)` pose une frontière : *ces* nœuds dedans, les autres dehors. Le
programme de ce notebook est de faire de ce choix un **objet mesurable** au lieu d'un présupposé :

1. **Faire varier la frontière** — même substrat, plusieurs découpages ; mesurer ce qui change et
   ce qui reste.
2. **La frontière qui maximise** — existe-t-il un découpage privilégié par une mesure donnée ?
   L'exhiber, ou montrer qu'il n'est pas unique.
3. **Le désaccord** — deux mesures privilégient-elles la même frontière ? Si non, c'est une
   dissociation à enregistrer.


## 0. Setup — configuration de `pyphi`

Conventions de la série (cf. IIT-3) : mono-cœur pour des exécutions déterministes, barres de
progression désactivées pour des sorties propres. On coupe aussi l'évaluation parallèle des
partitions : sur Windows, le *spawn* multiprocessing d'un moteur `pyphi` parallèle échoue en
exécution de notebook (`__main__` non protégé) — et nos réseaux de 4 nœuds n'en ont pas besoin.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import itertools
import pyphi
import numpy as np
import pandas as pd

pyphi.config.NUMBER_OF_CORES = 1
pyphi.config.PARALLEL_CUT_EVALUATION = False
pyphi.config.PROGRESS_BARS = False
pyphi.config.WELCOME_OFF = True

print("pyphi", pyphi.__version__)

pyphi 1.2.0


## 1. Le substrat — un réseau 4 nœuds, le même pour tout le notebook

Quatre nœuds logiques $A, B, C, D$ organisés en **deux paires d'échange** :

- $A$ prend l'état de $B$ au pas suivant, $B$ prend celui de $A$ (paire $A \leftrightarrow B$) ;
- $C$ et $D$ font de même entre eux (paire $C \leftrightarrow D$).

La structure « naturelle » saute aux yeux : deux paires. Le notebook montrera précisément que
cette évidence est un **présupposé de frontière** — et que la mesure $\Phi$ ne le confirme pas.

Le substrat est la chose physique — cette TPM, ces quatre nœuds. Il ne bougera plus d'une section
à l'autre ; seule la **frontière** (le choix des nœuds admis dans le sous-système) variera. C'est
le point méthodologique : on sépare le *substrat* (donné) du *découpage* (choisi).

In [2]:
def tpm_deux_paires():
    """TPM 4-noeuds : A<->B et C<->D s'echangeant leurs etats."""
    tpm = np.zeros((16, 4))
    for s in range(16):
        b = [(s >> (3 - i)) & 1 for i in range(4)]   # bits A,B,C,D
        tpm[s] = [b[1], b[0], b[3], b[2]]            # chaque paire s'echange
    return tpm

TPM = tpm_deux_paires()
CM = np.ones((4, 4))                    # connectivite complete
substrat = pyphi.Network(TPM, CM)
ETAT = (1, 1, 1, 1)                     # etat du reseau pour toutes les mesures dependantes de l'etat

print("Substrat : 4 noeuds en deux paires d'echange (A<->B, C<->D)")
print("Exemples de lignes TPM : (A,B,C,D)=(1,0,1,0) ->", TPM[int("1010", 2)].astype(int),
      " ; (1,0,0,1) ->", TPM[int("1001", 2)].astype(int))

Substrat : 4 noeuds en deux paires d'echange (A<->B, C<->D)
Exemples de lignes TPM : (A,B,C,D)=(1,0,1,0) -> [0 1 0 1]  ; (1,0,0,1) -> [0 1 1 0]


## 2. La frontière est un choix — l'énumérer

`pyphi.Subsystem(network, state, nodes)` matérialise une frontière : `nodes` dedans, le reste
dehors (conditionné, pas supprimé). Pour 4 nœuds, les frontières **non triviales** (au moins 2
nœuds — un nœud seul n'a rien à intégrer) sont au nombre de 11 : 6 paires, 4 triplets, le tout.
Les énumérer, c'est déjà admettre que « le système » n'est pas une donnée.

In [3]:
NOMS = "ABCD"
combos = [c for r in (2, 3, 4) for c in itertools.combinations(range(4), r)]
frontieres = ["".join(NOMS[i] for i in c) for c in combos]
print(f"{len(frontieres)} frontieres candidates (>= 2 noeuds) :")
print(", ".join(frontieres))

11 frontieres candidates (>= 2 noeuds) :
AB, AC, AD, BC, BD, CD, ABC, ABD, ACD, BCD, ABCD


## 3. Faire varier la frontière — ce qui change, ce qui reste

Pour chaque frontière, deux mesures sont calculées **sur le même substrat, dans le même état** :

- $\Phi$ **du sous-système** (`pyphi.compute.phi`) : mesure IIT 3.0 de l'irréductibilité de la
  frontière considérée — dépend de l'état ;
- **EI du sous-réseau induit** (information efficace de Hoel, `pyphi.macro.effective_info`) :
  la frontière lue comme un réseau **autonome** — indépendante de l'état. Cette lecture n'est
  possible que si la frontière ne coupe aucune dépendance en deux : si deux états du substrat qui
  projettent la même sous-configuration donnent des suites différentes, le sous-réseau n'est pas
  condensable — l'EI est **non définie** pour cette frontière (c'est déjà une donnée sur le bord).

C'est la table centrale du notebook : onze frontières, un seul substrat.

In [4]:
def phi_frontiere(noeuds_idx):
    """Phi IIT 3.0 du sous-systeme delimite par noeuds_idx, dans ETAT."""
    sub = pyphi.Subsystem(substrat, ETAT, tuple(noeuds_idx))
    return pyphi.compute.phi(sub)

def ei_induit(noeuds_idx):
    """EI (Hoel) du sous-reseau induit par la frontiere, si elle est condensable.

    Rend None quand la frontiere coupe une dependance : deux etats du substrat
    qui projettent la meme sous-configuration menent a des suites differentes —
    le bord n'est pas lisible comme reseau autonome.
    """
    k = len(noeuds_idx)
    lignes = {}
    for s in range(2 ** 4):
        proj = tuple((s >> (3 - i)) & 1 for i in noeuds_idx)   # sous-config courante
        nxt = tuple(int(TPM[s][j]) for j in noeuds_idx)        # sous-config suivante
        if proj in lignes and lignes[proj] != nxt:
            return None
        lignes[proj] = nxt
    sub_tpm = np.zeros((2 ** k, k))
    for proj, nxt in lignes.items():
        sub_tpm[int("".join(map(str, proj)), 2)] = nxt
    return pyphi.macro.effective_info(pyphi.Network(sub_tpm))

lignes = []
for c in combos:
    nom = "".join(NOMS[i] for i in c)
    ei = ei_induit(c)
    lignes.append({
        "Frontiere": nom,
        "Taille": len(c),
        "Phi (IIT 3.0, etat fixe)": round(phi_frontiere(c), 4),
        "EI induit (Hoel)": round(ei, 4) if ei is not None else "non definie (frontiere coupante)",
    })

table = pd.DataFrame(lignes)
table

,Frontiere,Taille,"Phi (IIT 3.0, etat fixe)",EI induit (Hoel)
0,AB,2,0.0,2.0
1,AC,2,1.0,non definie (frontiere coupante)
2,AD,2,0.0,non definie (frontiere coupante)
3,BC,2,0.0,non definie (frontiere coupante)
4,BD,2,1.0,non definie (frontiere coupante)
5,CD,2,0.0,2.0
6,ABC,3,0.0,non definie (frontiere coupante)
7,ABD,3,0.0,non definie (frontiere coupante)
8,ACD,3,0.0,non definie (frontiere coupante)
9,BCD,3,0.0,non definie (frontiere coupante)


**Ce qui change** : les deux colonnes — la frontière n'est pas une valeur neutre, elle
détermine ce qu'on voit. **Ce qui reste** : le substrat (TPM inchangée), l'état, la structure
causale. Le tableau est la preuve **calculée**, pas discutée, que des découpages distincts du même
substrat donnent des lectures différentes.

Trois faits saillants, tous mesurés :

- les paires « naturelles » $AB$ et $CD$ ont $\Phi = 0$ : parfaitement décomposables, l'évidence
  structurelle n'est pas l'irréductibilité ;
- les frontières **transversales** $AC$ et $BD$ — un nœud de chaque paire — portent $\Phi = 1$ :
  l'irréductibilité vit dans le croisement, pas dans la décomposition évidente ;
- $8$ frontières sur $11$ rendent l'EI **non définie** : elles coupent une dépendance, elles ne
  sont même pas lisibles comme réseaux autonomes. Le domaine des deux mesures n'est pas le même —
  l'écart fait partie du résultat.


## 4. La frontière qui maximise

L'**exclusion** est le postulat qui répond au problème de frontière en IIT : parmi les frontières
candidates, celle qui porte l'expérience est celle qui **maximise** $\Phi$ (le *complex*). Ce
n'est pas une déduction — c'est un axiome de la théorie, et le notebook le traite comme tel : une
mesure donnée **privilégie** une frontière, et cette préférence est elle-même mesurable.

Le maximiseur est-il unique ? Des **ex æquo** signifient que la mesure ne tranche pas — le choix
reste ouvert entre plusieurs bords, et c'est un résultat, pas un échec.

In [5]:
col_phi = "Phi (IIT 3.0, etat fixe)"
col_ei = "EI induit (Hoel)"

meilleur_phi = table[col_phi].max()
gagnants_phi = table.loc[table[col_phi] == meilleur_phi, "Frontiere"].tolist()

lisibles = table[table[col_ei].apply(lambda v: isinstance(v, float))]
meilleur_ei = lisibles[col_ei].max()
gagnants_ei = lisibles.loc[lisibles[col_ei] == meilleur_ei, "Frontiere"].tolist()

print(f"Max Phi = {meilleur_phi:.4f} atteint par : {gagnants_phi}"
      f"{'  (EX AEQUO — la mesure ne tranche pas)' if len(gagnants_phi) > 1 else ''}")
print(f"Max EI  = {meilleur_ei:.4f} atteint par : {gagnants_ei}"
      f"{'  (EX AEQUO — la mesure ne tranche pas)' if len(gagnants_ei) > 1 else ''}")
print(f"Domaine EI : {len(lisibles)}/{len(table)} frontieres lisibles comme reseaux autonomes")

Max Phi = 1.0000 atteint par : ['AC', 'BD']  (EX AEQUO — la mesure ne tranche pas)
Max EI  = 4.0000 atteint par : ['ABCD']
Domaine EI : 3/11 frontieres lisibles comme reseaux autonomes


## 5. Le désaccord — deux mesures, deux bords ?

Deux mesures, deux verdicts possibles sur la **même** question (où est la frontière qui compte ?).
Si elles privilégient la même frontière, elles concordent — résultat à enregistrer comme tel. Si
elles divergent, la dissociation est le résultat : le bord n'est pas dans le substrat, il est dans
la mesure qu'on choisit d'appliquer.

In [6]:
if set(gagnants_phi) & set(gagnants_ei):
    if set(gagnants_phi) == set(gagnants_ei):
        verdict = f"MEME FRONTIERE : Phi et EI induit privilegient {gagnants_phi}"
    else:
        verdict = f"RECROUVEMENT PARTIEL : {sorted(set(gagnants_phi) & set(gagnants_ei))}"
elif any(g not in lisibles["Frontiere"].tolist() for g in gagnants_phi):
    hors_domaine = [g for g in gagnants_phi if g not in lisibles["Frontiere"].tolist()]
    verdict = (f"FRONTIERES DIFFERENTES : Phi privilegie {gagnants_phi}, EI induit privilegie "
               f"{gagnants_ei} — et les favorites de Phi ({hors_domaine}) sont hors du domaine "
               f"de l'EI. Double dissociation : les mesures ne partagent ni le verdict ni le domaine.")
else:
    verdict = (f"FRONTIERES DIFFERENTES : Phi privilegie {gagnants_phi}, "
               f"EI induit privilegie {gagnants_ei} — dissociation a enregistrer")

print(verdict)
print()
print("Classement Phi (toutes frontieres) :")
print(table.sort_values(col_phi, ascending=False)[["Frontiere", col_phi]].to_string(index=False))
print()
print("Classement EI (frontieres lisibles seules) :")
print(lisibles.sort_values(col_ei, ascending=False)[["Frontiere", col_ei]].to_string(index=False))

FRONTIERES DIFFERENTES : Phi privilegie ['AC', 'BD'], EI induit privilegie ['ABCD'] — et les favorites de Phi (['AC', 'BD']) sont hors du domaine de l'EI. Double dissociation : les mesures ne partagent ni le verdict ni le domaine.

Classement Phi (toutes frontieres) :
Frontiere  Phi (IIT 3.0, etat fixe)
       AC                       1.0
       BD                       1.0
       AB                       0.0
       AD                       0.0
       BC                       0.0
       CD                       0.0
      ABC                       0.0
      ABD                       0.0
      ACD                       0.0
      BCD                       0.0
     ABCD                       0.0

Classement EI (frontieres lisibles seules) :
Frontiere EI induit (Hoel)
     ABCD              4.0
       AB              2.0
       CD              2.0


Le verdict de ce substrat est une **double dissociation**, et c'est le résultat le plus
instructif du notebook :

1. **Verdicts opposés** : $\Phi$ privilégie les frontières transversales ($AC$, $BD$), l'EI
   induit privilégie le tout ($ABCD$) — deux réponses différentes à « où est la frontière qui
   compte ? ».
2. **Domaines disjoints sur les favorites** : les frontières préférées de $\Phi$ coupent une
   dépendance — l'EI ne peut même pas les évaluer. Chaque mesure répond depuis son domaine, et
   les domaines ne se recouvrent pas sur les candidats décisifs.

La leçon ne va pas dans le sens de « l'une a raison » : $\Phi$ (irréductibilité d'un
sous-système dans un état, le reste conditionné) et l'EI induit (dynamique autonome d'un
sous-réseau, tout état confondu) posent deux questions différentes au même substrat. Le bord
n'est pas dans le substrat : il est dans la **mesure qu'on choisit d'appliquer** — et ce choix-là,
personne ne le fait à notre place. C'est exactement ce que le problème de frontière voulait rendre
visible.


## 6. Exercices

Trois exercices sur le même socle méthodologique — chacun sur **votre** substrat. Les stubs
s'exécutent sans erreur (consigne imprimée) ; compléter, pas remplacer.


### Exercice 1 — Faire varier la frontière, sur votre substrat

Construire un autre réseau 4 nœuds (autre topologie : chaîne, deux paires, anneau...) et refaire
la table de la section 3 : au moins trois frontières calculées, avec $\Phi$ et EI induit. Noter
ce qui change par rapport au noyau majoritaire + suiveur.

In [7]:
# Exercice 1 : votre substrat
# Etape 1 : definir tpm_votre() (4 noeuds, autre logique que majoritaire+suiveur).
# Etape 2 : choisir 3+ frontieres et calculer phi_frontiere / ei_induit dessus.
# Indice : reprendre la cellule de la section 3 en changeant seulement la TPM.

def tpm_votre():
    pass  # TODO etudiant
    return None

print("Exercice a completer : tpm_votre() + table de 3+ frontieres (Phi, EI induit).")

Exercice a completer : tpm_votre() + table de 3+ frontieres (Phi, EI induit).


### Exercice 2 — La frontière qui maximise, sur votre substrat

Sur votre substrat : quelles frontières maximisent $\Phi$ ? Le maximiseur est-il unique, ou y
a-t-il ex æquo ? Que se passe-t-il si vous changez l'état (`ETAT`) — le maximiseur bouge-t-il ?

In [8]:
# Exercice 2 : le maximiseur
# Etape 1 : reprendre votre table et extraire argmax(Phi) comme la section 4.
# Etape 2 : refaire le calcul dans un autre etat (ex. (0,0,0,0) puis (1,0,1,0)).
# Indice : Phi depend de l'etat ; EI non. Un maximiseur qui bouge avec l'etat
#          dit que la frontiere privilegiee n'est meme pas stable dans le temps.

print("Exercice a completer : argmax(Phi) dans 2 etats distincts, verdict unicite/stabilite.")

Exercice a completer : argmax(Phi) dans 2 etats distincts, verdict unicite/stabilite.


### Exercice 3 — Le désaccord, verdict explicite

Sur votre substrat : $\Phi$ et EI induit privilégient-ils la même frontière ? Conclure par un
verdict explicite parmi : **même frontière / frontières différentes / indécidable dans ce
budget**. Si frontières différentes : enregistrer la dissociation (frontière $\Phi$, frontière
EI, écart des classements).

In [9]:
# Exercice 3 : le verdict
# Etape 1 : extraire gagnants_phi et gagnants_ei de VOTRE table (section 4).
# Etape 2 : produire le verdict — meme frontiere / frontieres differentes / indecidable.
# Indice : "indecidable" est legitime si les calculs ne tiennent pas dans le budget ;
#          mais il faut l'avoir tente pour le dire.

print("Exercice a completer : verdict explicite (meme frontiere / differentes / indecidable).")

Exercice a completer : verdict explicite (meme frontiere / differentes / indecidable).


## Conclusion

| Question | Réponse de ce notebook | Où |
|---|---|---|
| Qui décide des bords ? | La frontière est un argument (`nodes`) de `Subsystem` — un choix explicite, jamais une donnée du substrat | sections 2-3 |
| Ce qui change avec le bord | $\Phi$ et EI : calculés pour 11 frontières d'un même substrat — table, pas discussion | section 3 |
| Un bord privilégié ? | Le maximiseur de chaque mesure est extrait ; l'ex æquo est un résultat (la mesure ne tranche pas) | section 4 |
| Deux mesures, un bord ? | Verdict explicite imprimé — la dissociation éventuelle est le résultat enregistré | section 5 |

Le problème de frontière ne se résout pas en choisissant la bonne mesure — il se **montre** :
chaque mesure répond à sa manière, et l'écart entre les réponses est une propriété du couple
(substrat, mesure), pas du substrat seul. C'est la dette que toute analyse de recollement porte
en amont d'elle-même (cf. #12206) : mesurer un défaut de recollement mesure aussi le choix
arbitraire du découpage — sauf à rendre ce choix visible d'abord.


## Références

- Mayner WGP, Marshall W, Albantakis L, Findlay G, Marchman R, Tononi G. (2018). *PyPhi: A
  toolbox for integrated information theory.* PLOS Computational Biology 14(7): e1006343.
- Oizumi M., Albantakis L., Tononi G. (2014). *From the phenomenology to the mechanisms of
  consciousness: Integrated Information Theory 3.0.* PLoS Computational Biology.
- Hoel E. P., Albantakis L., Tononi G. (2013). *Quantifying causal integration and power.*
  Proceedings of the National Academy of Sciences.
- Gómez Emilsson A. — *The Boundary Problem* (Qualia Computing) : **proposition** d'une
  réponse au problème de frontière (topologie de champs électromagnétiques), citée ici comme
  piste à évaluer, pas comme résultat établi — le présent notebook mesure le **problème**, il
  n'adopte aucune de ses réponses candidates.
